In [ ]:
import os
import base64
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep

# === Load Tokens ===
load_dotenv("D:/Android_Mobile_App/AndroidProject_dataset/All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found.")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

# === Functions ===
def extract_owner_repo(url):
    try:
        parts = url.split("github.com/")[-1].replace(".git", "").split("/")
        return parts[0], parts[1]
    except:
        return None, None

def check_manifest_and_activity(owner, repo):
    search_url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{owner}/{repo}"
    while True:
        res = requests.get(search_url, headers=get_headers())
        if res.status_code == 403:
            rotate_token()
            sleep(1)
            continue
        if res.status_code != 200:
            return "no", "no"
        break

    data = res.json()
    if data.get("total_count", 0) == 0:
        return "no", "no"

    manifest_url = data["items"][0]["url"]
    while True:
        res = requests.get(manifest_url, headers=get_headers())
        if res.status_code == 403:
            rotate_token()
            sleep(1)
            continue
        if res.status_code != 200:
            return "yes", "no"
        break

    try:
        content = res.json().get("content")
        if content:
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore")
            root = ET.fromstring(decoded)
            has_activity = root.findall(".//activity")
            return "yes", "yes" if has_activity else "no"
    except:
        return "yes", "no"
    return "yes", "no"

# === Load CSV ===
input_path = "D:/Android_Mobile_App/AndroidProject_dataset/Repo_List_OR.csv"
df = pd.read_csv(input_path)
df["has_manifest"] = "no"
df["has_activity"] = "no"

# === Process Each Repo ===
for i, row in df.iterrows():
    owner, repo = extract_owner_repo(row["html_url"])
    if owner and repo:
        manifest, activity = check_manifest_and_activity(owner, repo)
        df.at[i, "has_manifest"] = manifest
        df.at[i, "has_activity"] = activity

# === Save Output ===
output_path = "D:/Android_Mobile_App/AndroidProject_dataset/Repo_List_OR_with_manifest.csv"
df.to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")
